# MambaCNNLite Multi-Seed Training

MambaCNNLite - Rice Disease Detection (sub-1 MB model).
**Trains on multiple seeds and reports the best results.**

Architecture: Depthwise-Separable CNN Stem -> Lite Mamba S6 blocks -> classify

Target: < 1 MB on-disk weights (~70K params)

## Environment Setup

In [ ]:
!pip install onnxscript

In [ ]:
import os
import sys

os.environ["PYTHONHASHSEED"] = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

if 'torch' in sys.modules:
    print("⚠️  WARNING: torch was already imported by Kaggle.")
    print("   For strict determinism, add CUBLAS_WORKSPACE_CONFIG=:4096:8")
    print("   in Kaggle Settings → Add-ons → Environment Variables")

## Imports

In [ ]:
import random
import numpy as np
import warnings
import json
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings('ignore')

## Configuration

In [ ]:
MODEL_NAME = "mamba_cnn_lite"

# ============================================
# MULTI-SEED CONFIGURATION
# ============================================
SEEDS = [0, 1, 42, 99, 123, 456, 789, 1024]  # List of seeds to try

# Data configuration
SPLIT_SEED      = 42              # Keep fixed for fair comparison
DATASET_PATH    = "/kaggle/input/datasets/anshulm257/rice-disease-dataset/Rice_Leaf_AUG"

# Training hyperparameters
IMAGE_SIZE      = 96              # Smaller for lite model
BATCH_SIZE      = 128
EPOCHS          = 80
LR              = 2e-3
WEIGHT_DECAY    = 0.05
LABEL_SMOOTHING = 0.05
PATIENCE        = 20
NUM_WORKERS     = 4
D_MODEL         = 64
N_MAMBA         = 2
D_STATE         = 8               # Lite: half the state size

# Output
OUTPUT_DIR      = "outputs"
GPUS            = ""

print(f"Will train with {len(SEEDS)} seeds: {SEEDS}")

## Model Definition

In [ ]:
class DepthwiseSeparable(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.dw = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, kernel_size=3, padding=1, groups=in_ch, bias=False),
            nn.BatchNorm2d(in_ch), nn.GELU())
        self.pw = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU())
    def forward(self, x): return self.pw(self.dw(x))


class MambaBlockLite(nn.Module):
    def __init__(self, d_model: int, d_state: int = 8, d_conv: int = 4, expand: int = 2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_inner = d_model * expand
        self.dt_rank = max(1, self.d_inner // 16)
        self.norm = nn.LayerNorm(d_model)
        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)
        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=d_conv,
                                 padding=d_conv - 1, groups=self.d_inner, bias=True)
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + 2 * d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)
        A_init = torch.log(torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).repeat(self.d_inner, 1))
        self.A_log = nn.Parameter(A_init)
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def _selective_scan(self, x):
        B, L, _ = x.shape
        A = -torch.exp(self.A_log.float())
        D = self.D.float()
        xBC = self.x_proj(x)
        dt_raw, B_p, C = xBC.split([self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = F.softplus(self.dt_proj(dt_raw))
        dA = torch.exp(torch.einsum('bld,ds->blds', dt, A))
        dB = torch.einsum('bld,bls->blds', dt, B_p)
        h = torch.zeros(B, self.d_inner, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for i in range(L):
            h = dA[:, i] * h + dB[:, i] * x[:, i, :, None]
            ys.append(torch.einsum('bds,bs->bd', h, C[:, i]))
        return torch.stack(ys, dim=1) + x * D.to(x.dtype)

    def forward(self, x):
        residual = x
        x = self.norm(x)
        xz = self.in_proj(x)
        x_h, z = xz.chunk(2, dim=-1)
        x_h = self.conv1d(x_h.transpose(1, 2))[:, :, :x_h.shape[1]].transpose(1, 2)
        x_h = F.silu(x_h)
        y = self._selective_scan(x_h)
        return self.out_proj(y * F.silu(z)) + residual


class MambaCNNLite(nn.Module):
    def __init__(self, num_classes: int, d_model: int = 64, n_mamba: int = 2, d_state: int = 8):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(16), nn.GELU(), nn.MaxPool2d(2),
            DepthwiseSeparable(16, 32), nn.MaxPool2d(2),
            DepthwiseSeparable(32, d_model), nn.MaxPool2d(2),
            DepthwiseSeparable(d_model, d_model), nn.MaxPool2d(2))
        self.mamba = nn.Sequential(*[MambaBlockLite(d_model, d_state=d_state) for _ in range(n_mamba)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(d_model, num_classes))

    def forward(self, x):
        x = self.stem(x)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        x = self.mamba(x)
        x = self.norm(x)
        return self.head(x.mean(dim=1))

## Helper Functions

In [ ]:
def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    cublas_config = os.environ.get("CUBLAS_WORKSPACE_CONFIG", "")
    if cublas_config in [":4096:8", ":16:8"]:
        torch.use_deterministic_algorithms(True, warn_only=False)
    else:
        torch.use_deterministic_algorithms(True, warn_only=True)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [ ]:
def make_loaders(dataset_path, image_size, batch_size, num_workers, split_seed, seed):
    mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    train_tf = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
        transforms.RandomRotation(20),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.15),
        transforms.ToTensor(), transforms.Normalize(mean, std)])
    eval_tf = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(), transforms.Normalize(mean, std)])
    
    train_dataset = datasets.ImageFolder(dataset_path, transform=train_tf)
    eval_dataset = datasets.ImageFolder(dataset_path, transform=eval_tf)
    class_names = train_dataset.classes
    n = len(train_dataset)
    indices = torch.randperm(n, generator=torch.Generator().manual_seed(split_seed)).tolist()
    n_train, n_val = int(0.8 * n), int(0.1 * n)
    train_ds = Subset(train_dataset, indices[:n_train])
    val_ds = Subset(eval_dataset, indices[n_train:n_train + n_val])
    test_ds = Subset(eval_dataset, indices[n_train + n_val:])
    
    g = torch.Generator().manual_seed(seed)
    kw = dict(batch_size=batch_size, num_workers=num_workers, pin_memory=True)
    train_loader = DataLoader(train_ds, shuffle=True, generator=g, worker_init_fn=seed_worker, **kw)
    val_loader = DataLoader(val_ds, shuffle=False, **kw)
    test_loader = DataLoader(test_ds, shuffle=False, **kw)
    return train_loader, val_loader, test_loader, class_names

In [ ]:
def run_epoch(model, loader, criterion, optimizer, device, train=True):
    model.train(train)
    total_loss = correct = total = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if train: optimizer.zero_grad()
            logits = model(imgs)
            loss = criterion(logits, labels)
            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    return total_loss / total, correct / total

In [ ]:
def train_single_seed(seed, device, seed_output_dir):
    """Train MambaCNNLite with a single seed and return results."""
    print(f"\n{'='*60}")
    print(f"  Training with SEED: {seed}")
    print(f"{'='*60}")
    
    seed_everything(seed)
    os.makedirs(seed_output_dir, exist_ok=True)
    
    train_loader, val_loader, test_loader, class_names = make_loaders(
        DATASET_PATH, IMAGE_SIZE, BATCH_SIZE, NUM_WORKERS, SPLIT_SEED, seed)
    num_classes = len(class_names)
    
    model = MambaCNNLite(num_classes=num_classes, d_model=D_MODEL, n_mamba=N_MAMBA, d_state=D_STATE).to(device)
    total_p = sum(p.numel() for p in model.parameters())
    size_kb = total_p * 4 / 1024
    print(f"  Model params: {total_p:,} ({size_kb:.1f} KB)")
    
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    best_val_loss = float('inf')
    best_weights = None
    patience_cnt = 0
    best_ckpt = os.path.join(seed_output_dir, f"best_{MODEL_NAME}_seed{seed}.pth")
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
        vl_loss, vl_acc = run_epoch(model, val_loader, criterion, None, device, train=False)
        scheduler.step()
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)
        
        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:>3}: Train Acc={tr_acc:.4f}, Val Acc={vl_acc:.4f}")
        
        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
            torch.save(best_weights, best_ckpt)
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print(f"  Early stopping at epoch {epoch}")
                break
    
    model.load_state_dict(best_weights)
    test_loss, test_acc = run_epoch(model, test_loader, criterion, None, device, train=False)
    
    print(f"\n📊 Seed {seed} → Test Accuracy: {test_acc*100:.2f}%")
    
    return {
        "seed": seed,
        "test_acc": test_acc * 100,
        "test_loss": test_loss,
        "best_val_loss": best_val_loss,
        "best_ckpt": best_ckpt,
        "history": history,
        "class_names": class_names,
        "model_size_kb": size_kb,
    }

## Setup

In [ ]:
if GPUS:
    os.environ["CUDA_VISIBLE_DEVICES"] = GPUS
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Device: {device}")

## Multi-Seed Training Loop

In [ ]:
all_results = []

print(f"\n{'#'*60}")
print(f"  STARTING MULTI-SEED TRAINING")
print(f"  Seeds: {SEEDS}")
print(f"{'#'*60}")

for i, seed in enumerate(SEEDS):
    print(f"\n[{i+1}/{len(SEEDS)}] Training with seed {seed}...")
    seed_output_dir = os.path.join(OUTPUT_DIR, "runs", f"seed_{seed}")
    
    result = train_single_seed(seed, device, seed_output_dir)
    all_results.append(result)
    
    with open(os.path.join(OUTPUT_DIR, "all_seed_results.json"), "w") as f:
        json.dump([{k: v for k, v in r.items() if k not in ['history', 'class_names']} for r in all_results], f, indent=2)

print(f"\n{'#'*60}")
print(f"  MULTI-SEED TRAINING COMPLETE")
print(f"{'#'*60}")

## Results Comparison

In [ ]:
df = pd.DataFrame([{k: v for k, v in r.items() if k not in ['history', 'class_names']} for r in all_results])
df = df.sort_values("test_acc", ascending=False).reset_index(drop=True)

print("\n" + "="*70)
print("  RESULTS SUMMARY (Sorted by Test Accuracy)")
print("="*70)
print(df[["seed", "test_acc", "test_loss", "model_size_kb"]].to_string(index=False))
print("="*70)

print(f"\n📊 Statistics:")
print(f"   Test Accuracy: Mean={df['test_acc'].mean():.2f}%, Std={df['test_acc'].std():.2f}%")
print(f"   Best seed: {df.iloc[0]['seed']} with Acc: {df.iloc[0]['test_acc']:.2f}%")
print(f"   Model size: {df.iloc[0]['model_size_kb']:.1f} KB (Under 1MB: {'YES' if df.iloc[0]['model_size_kb'] < 1024 else 'NO'})")

## Best Model Export

In [ ]:
best_result = all_results[SEEDS.index(int(df.iloc[0]['seed']))]
best_seed = int(df.iloc[0]['seed'])
best_ckpt = best_result['best_ckpt']
class_names = best_result['class_names']

print(f"\n🏆 BEST MODEL")
print(f"   Seed: {best_seed}")
print(f"   Test Accuracy: {df.iloc[0]['test_acc']:.2f}%")

# Load best model and export to ONNX
train_loader, val_loader, test_loader, _ = make_loaders(
    DATASET_PATH, IMAGE_SIZE, BATCH_SIZE, NUM_WORKERS, SPLIT_SEED, best_seed)
model = MambaCNNLite(num_classes=len(class_names), d_model=D_MODEL, n_mamba=N_MAMBA, d_state=D_STATE).to(device)
model.load_state_dict(torch.load(best_ckpt))
model.eval()

# ONNX export
onnx_path = best_ckpt.replace(".pth", ".onnx")
dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
torch.onnx.export(model, dummy, onnx_path, input_names=["input"], output_names=["output"],
                  dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}}, opset_version=17)

ckpt_kb = os.path.getsize(best_ckpt) / 1024
onnx_kb = os.path.getsize(onnx_path) / 1024
print(f"✅ Checkpoint: {best_ckpt} ({ckpt_kb:.1f} KB)")
print(f"✅ ONNX: {onnx_path} ({onnx_kb:.1f} KB)")
print(f"   Under 1 MB: {'YES' if ckpt_kb < 1024 else 'NO'} (pth) | {'YES' if onnx_kb < 1024 else 'NO'} (onnx)")

## Best Model Confusion Matrix

In [ ]:
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        logits = model(imgs.to(device))
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title(f'Confusion Matrix (Seed {best_seed})')
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title('Normalized')
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_best_confusion_matrix.png")
plt.savefig(cm_path, dpi=150)
plt.show()

## Save Summary

In [ ]:
summary = {
    "model": MODEL_NAME,
    "seeds_tested": SEEDS,
    "best_seed": best_seed,
    "best_test_acc": float(df.iloc[0]['test_acc']),
    "mean_test_acc": float(df['test_acc'].mean()),
    "std_test_acc": float(df['test_acc'].std()),
    "model_size_kb": float(df.iloc[0]['model_size_kb']),
    "best_ckpt": best_ckpt,
    "onnx_path": onnx_path,
}
with open(os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
df.to_csv(os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_seed_results.csv"), index=False)

print(f"\n{'='*70}")
print(f"  FINAL SUMMARY")
print(f"{'='*70}")
print(f"  Seeds tested   : {SEEDS}")
print(f"  Best seed      : {best_seed}")
print(f"  Best Test Acc  : {df.iloc[0]['test_acc']:.2f}%")
print(f"  Mean Test Acc  : {df['test_acc'].mean():.2f}% (±{df['test_acc'].std():.2f}%)")
print(f"  Model size     : {df.iloc[0]['model_size_kb']:.1f} KB")
print(f"  Best model     : {best_ckpt}")
print(f"  ONNX model     : {onnx_path}")
print(f"{'='*70}")